# Mass-Spring-Damper Forecasting — a Physics-Based Approach

This notebook is the worked example that goes beyond the quickstart's persistence baseline. The idea is simple and powerful: the `mass_spring_damper` twin simulates a *known* physical system — a mass on a spring with a viscous damper, driven by a sinusoidal force — so instead of treating the sensor stream as an anonymous time series, we can **estimate the physical parameters from the data and integrate the equation of motion forward** to produce the forecast. When the physics is right, this approach is nearly unbeatable, because your model and the simulator are solving the same differential equation.

The governing equation is

$$m\,\ddot{x} + c\,\dot{x} + k\,x = F(t), \qquad F(t) = 0.5\,\sin(2\pi t)$$

with mass $m$, damping $c$, stiffness $k$. The mass is known from the system specification ($m = 1$ kg); the contest may have degraded $k$ or $c$ through faults, and the sensors add noise — so $k$ and $c$ must be estimated from what we observe.

The plan: collect live data → estimate $k$ and $c$ by least squares → validate the fit → integrate forward `eval_steps` steps with a Runge-Kutta scheme → submit the configured **target variables**.

## Installation — run this cell once

In [ ]:
%pip install "epic-elios-client[notebook]" matplotlib numpy

## Configure the Client

We fix the known mass here. Everything else — sampling rate, number of evaluation steps, target variables — will be read from the contest itself, so the notebook adapts to any mass-spring-damper contest without edits.

In [ ]:
from epic_client import EPICClient

SERVER_URL = "https://epic.elioslab.net"

# Mass is known from the system specification.
MASS = 1.0

client = EPICClient(SERVER_URL)

## 1. Log In

Use the credentials from your invitation-link registration (username = your email) or the account your instructor created. The client keeps the JWT and re-authenticates every call; if requests start failing after a long session, the token expired — just log in again.

In [ ]:
client.login("your-username", "your-password")

## 2. Browse Available Contests

Look for a contest based on the `mass_spring_damper` twin. Remember that this listing is **personalized by visibility**: PUBLIC contests are visible to everyone, while PRIVATE and INVITATION_ONLY ones appear only if the organizer invited your email (or you are already registered). The two columns that define your deliverable are `eval_steps` — how many values to predict per variable — and `target_variables` — which variables are scored. This notebook can forecast `position` and `velocity`, since both follow from integrating the equation of motion.

In [ ]:
import pandas as pd

contests = client.list_contests(status="ACTIVE")

rows = []
for c in contests:
    task_cfg = c.get("tasks", [{}])[0].get("configuration", {})
    rows.append({
        "contest_id":                c.get("contest_id"),
        "name":                      c.get("name"),
        "visibility":                c.get("visibility"),
        "twin_id":                   c.get("twin_id"),
        "sampling_rate_hz":          c.get("sampling_rate_hz"),
        "end_of_observation":        c.get("end_of_observation"),
        "prediction_horizon_seconds": c.get("prediction_horizon_seconds"),
        "eval_steps":                task_cfg.get("eval_steps"),
        "target_variables":          ", ".join(task_cfg.get("target_variables", [])),
    })

pd.DataFrame(rows)

Copy a `contest_id` from the table above. The cell below derives the integration time step `DT` from the sampling rate — this is the same step the simulator itself uses, which is exactly what we want for the forward integration later.

In [ ]:
CONTEST_ID = "replace-with-contest-id"

# Read contest parameters.
contest = next(c for c in contests if c["contest_id"] == CONTEST_ID)
task_config = contest["tasks"][0]["configuration"]
SAMPLING_RATE_HZ = contest["sampling_rate_hz"]
DT = 1.0 / SAMPLING_RATE_HZ
EVAL_STEPS = task_config["eval_steps"]
TARGET_VARIABLES = task_config.get("target_variables") or ["position"]

print(f"sampling_rate_hz : {SAMPLING_RATE_HZ} Hz  (dt = {DT:.4f} s)")
print(f"eval_steps       : {EVAL_STEPS}")
print(f"target_variables : {TARGET_VARIABLES}")
print(f"evaluation window: {EVAL_STEPS * DT:.1f} s  ({EVAL_STEPS} steps × {DT:.4f} s)")

## 3. Register for the Contest

For a PUBLIC contest this always succeeds. For a PRIVATE or INVITATION_ONLY contest, registration is accepted only because the organizer invited your email — and if you created your account through the invitation link itself, you are already registered and this call simply confirms it.

In [ ]:
client.register(CONTEST_ID)

## 4. Collect Live Observations — Observation Phase

Parameter estimation by finite differences is sensitive to how much signal you have: collect at least a few full oscillation periods (the nominal system has a natural frequency near $\sqrt{k/m}/2\pi \approx 0.5$ Hz, so 60 seconds gives ~30 periods). The CSV mirror means a kernel restart doesn't cost you the data. Remember the stream only exists during the observation phase, gaps in `sequence_id` mean lost (unrecoverable) readings, and what you receive are *noisy sensor measurements*, not the clean latent state — our least-squares fit will average that noise out.

In [ ]:
observations = await client.collect(
    CONTEST_ID,
    duration_seconds=60,          # adjust to match the observation window
    csv_path="msd_observations.csv",
)
print(f"Collected {len(observations)} observations  ({len(observations) * DT:.1f} s)")

## 5. Build the Dataset

Flatten the observations into a DataFrame and add a time axis derived from `sequence_id` (which counts simulator steps, so consecutive ids are exactly `DT` apart — more reliable than wall-clock timestamps).

In [ ]:
import numpy as np

rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp":   obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df["t"] = (df["sequence_id"] - df["sequence_id"].iloc[0]) * DT
df.head()

## 6. Plot the Position Signal

You should see a clean sinusoid-like oscillation, possibly with visible noise. If the amplitude or period drifts over time, a **fault** is probably active — degrading stiffness or damping mid-contest — which makes constant-parameter estimation an approximation (see *Next Steps* for how to handle that).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(df["t"], df["position"], linewidth=1.2, label="position")
plt.xlabel("Time (s)")
plt.ylabel("Position (m)")
plt.title("Live position stream — Mass-Spring-Damper")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Estimate the System Parameters

We discretise the equation of motion. From the position samples alone we reconstruct velocity and acceleration with **central finite differences**:

$$v_i \approx \frac{x_{i+1} - x_{i-1}}{2\,\Delta t}, \qquad a_i \approx \frac{x_{i+1} - 2x_i + x_{i-1}}{\Delta t^2}$$

Rearranging $m a = F - kx - cv$ gives, at every interior sample, one linear equation in the two unknowns $(k, c)$:

$$k\,x_i + c\,v_i = F(t_i) - m\,a_i$$

Stacking all samples produces an overdetermined system solved by **least squares** — which is also what makes the estimate robust to sensor noise: hundreds of noisy equations average out to accurate parameters. Compare the estimates with the nominal values ($k=10$, $c=0.5$); a clear deviation usually means the organizer scheduled a fault.

In [ ]:
x = df["position"].values
t = df["t"].values

# Central finite differences (exclude first and last points)
v = (x[2:] - x[:-2]) / (2 * DT)
a = (x[2:] - 2 * x[1:-1] + x[:-2]) / (DT ** 2)
x_mid = x[1:-1]
t_mid = t[1:-1]

# External forcing F(t) = 0.5 * sin(2π * t)
F = 0.5 * np.sin(2 * np.pi * t_mid)

# Right-hand side: F(t) - m*a
b = F - MASS * a

# Design matrix [x, v]
A = np.column_stack([x_mid, v])

# Least squares: solve for [k, c]
params, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
k_est, c_est = params

print(f"Estimated stiffness  k = {k_est:.4f}  (nominal: 10.0)")
print(f"Estimated damping    c = {c_est:.4f}  (nominal:  0.5)")

## 8. Validate the Fit

Before trusting the parameters, reconstruct the acceleration from the fitted model and overlay it on the measured one. A good fit tracks the measured curve closely with small RMSE; a systematic mismatch (growing error, phase shift) suggests the parameters changed during the observation window — a time-varying fault — in which case fitting on only the most recent portion of the data usually helps.

In [ ]:
a_pred = (F - k_est * x_mid - c_est * v) / MASS

plt.figure(figsize=(12, 4))
plt.plot(t_mid, a, label="measured (finite diff)", linewidth=1.2)
plt.plot(t_mid, a_pred, label="model fit", linewidth=1.2, linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("Acceleration (m/s²)")
plt.title("Parameter fit validation")
plt.legend()
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((a - a_pred) ** 2))
print(f"Fit RMSE (acceleration): {rmse:.6f} m/s²")

## 9. Forecast the Full Evaluation Window

Now the payoff: with $(\hat{k}, \hat{c})$ in hand, we integrate the equation of motion forward from the last observed state, one `DT` step at a time, for exactly `eval_steps` steps — producing one predicted value per evaluation step, which is precisely the shape the scorer expects. We use a classical **fourth-order Runge-Kutta** integrator: at 10–20 Hz its per-step error is far below the sensor noise floor, so the forecast quality is limited by the parameter estimates, not the integration.

Note that integrating the state forward yields *both* position and velocity at every step — convenient, because either or both may be in the contest's `target_variables`.

In [ ]:
def msd_rhs(t, x, v, k, c, m):
    F = 0.5 * np.sin(2 * np.pi * t)
    return v, (F - c * v - k * x) / m


def rk4_step(t, x, v, dt, k, c, m):
    k1x, k1v = msd_rhs(t,        x,              v,              k, c, m)
    k2x, k2v = msd_rhs(t + dt/2, x + dt/2*k1x,  v + dt/2*k1v,  k, c, m)
    k3x, k3v = msd_rhs(t + dt/2, x + dt/2*k2x,  v + dt/2*k2v,  k, c, m)
    k4x, k4v = msd_rhs(t + dt,   x + dt*k3x,    v + dt*k3v,    k, c, m)
    x_new = x + (dt / 6) * (k1x + 2*k2x + 2*k3x + k4x)
    v_new = v + (dt / 6) * (k1v + 2*k2v + 2*k3v + k4v)
    return x_new, v_new


# Initial state from the last two observations.
x0 = x[-1]
v0 = (x[-1] - x[-2]) / DT
t0 = t[-1]

# Integrate forward for eval_steps steps — one predicted value per step.
x_curr, v_curr, t_curr = x0, v0, t0
position_forecast = []
velocity_forecast = []

for _ in range(EVAL_STEPS):
    x_curr, v_curr = rk4_step(t_curr, x_curr, v_curr, DT, k_est, c_est, MASS)
    t_curr += DT
    position_forecast.append(x_curr)
    velocity_forecast.append(v_curr)

print(f"Forecast length : {len(position_forecast)} steps  (expected {EVAL_STEPS})")
print(f"Position first 5: {[round(v, 4) for v in position_forecast[:5]]}")
print(f"Position last  5: {[round(v, 4) for v in position_forecast[-5:]]}")

## 10. Plot the Forecast

Visualise the forecast continuing the observed history past the dotted line (the end of your collection). It should look like a seamless continuation of the oscillation — if there's a kink at the boundary, the initial velocity estimate from the last two samples was noisy; estimating it from a short finite-difference window instead is an easy refinement.

In [ ]:
t_forecast = t[-1] + DT * (np.arange(1, EVAL_STEPS + 1))

plt.figure(figsize=(12, 4))
plt.plot(t, x, label="observed", linewidth=1.2)
plt.plot(t_forecast, position_forecast, label=f"forecast ({EVAL_STEPS} steps)",
         linewidth=1.5, linestyle="--", color="tab:orange")
plt.axvline(t[-1], color="gray", linestyle=":", label="observation end")
plt.xlabel("Time (s)")
plt.ylabel("Position (m)")
plt.title("Mass-Spring-Damper — Observation History + Forecast")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Build and Submit the Prediction

The payload must contain **every target variable** declared by the contest, each with **exactly `eval_steps`** values. The cell maps our integrated state onto the requested targets and refuses politely if the contest asks for something this physics model doesn't produce (for example `temperature`, which on this twin only matters under the friction fault).

Remember submissions are only accepted after the evaluation window has fully elapsed — a `409` here just means *wait until the window opens, then re-run*. By then, your forecast is genuinely prospective: the ground truth was recorded while nobody could see it.

In [ ]:
supported_targets = {
    "position": position_forecast,
    "velocity": velocity_forecast,
}
unsupported_targets = [target for target in TARGET_VARIABLES if target not in supported_targets]
if unsupported_targets:
    raise ValueError(
        "This example can forecast position and velocity only. "
        f"The selected contest requires unsupported targets: {unsupported_targets}"
    )

payload = {
    "forecast": {
        target: [float(v) for v in supported_targets[target]]
        for target in TARGET_VARIABLES
    }
}

print("Payload target lengths:", {k: len(v) for k, v in payload["forecast"].items()})

submission = client.submit(
    contest_id=CONTEST_ID,
    task_id="forecasting",
    payload=payload,
)
print("Submission:", submission)

## 12. Check Scores and the Leaderboard

Scoring is asynchronous: the submission starts `PENDING` and becomes `EVALUATED` with one MAE score per target variable (lower is better, in the variable's own units — metres for position). You are scored against the **clean ground truth**, so the noise you fought during estimation does not contaminate the evaluation. The leaderboard keeps your best submission, so iterate freely.

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

print("=== My Submissions ===")
for sub in scores.get("submissions", []):
    print(f"  {sub['submission_id'][:8]}…  status={sub['status']}  "
          f"scores={[round(s['value'], 4) for s in sub.get('scores', [])]}")

print("\n=== Leaderboard ===")
for entry in leaderboard.get("entries", []):
    print(f"  Rank {entry['rank']}  user={entry['user_id'][:8]}…  score={entry['score']:.4f}")

## Next Steps

This approach estimates constant $k$ and $c$ over the full observation window — its main weakness when a contest schedules **gradual faults** that change the parameters over time. Three refinements, in increasing ambition: fit on a *sliding window* and use the most recent estimates; fit a *time-varying* parameter model ($k(t)$ linear or exponential) and extrapolate it through the evaluation window; or go hybrid — use this physics forecast as a baseline and train a small ML model on its residuals to capture whatever the physics misses. And if the contest's `target_variables` include something outside this model, like `temperature`, that is your cue to combine this notebook with a data-driven approach from the quickstart.